# HSC x Legacy Survey Cross-Match Demo

This notebook demonstrates how to load cross-matched HSC and Legacy Survey data using the `CrossMatchedDataLoader`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mmu.datapipes import CrossMatchedDataLoader

In [ ]:
HSC_PATH = '/mnt/ceph/users/polymathic/MultimodalUniverse/hsc/'
LEGACYSURVEY_PATH = '/mnt/ceph/users/polymathic/MultimodalUniverse/legacysurvey/'

loader = CrossMatchedDataLoader(
    left_dataset_path=HSC_PATH,
    right_dataset_path=LEGACYSURVEY_PATH,
    batch_size=4,
    num_workers=0,
)

loader.setup(stage='fit')
train_dl = loader.train_dataloader()

In [ ]:
batch = next(iter(train_dl))

print(f"Object IDs: {batch['object_id']}")
print(f"HSC flux shape: {np.array(batch['hsc_image']['flux']).shape}  # (batch, bands, H, W)")
print(f"Legacy Survey flux shape: {np.array(batch['legacysurvey_image']['flux']).shape}")

In [ ]:
def make_rgb(flux, percentile=(1, 99)):
    """Create RGB image from multi-band flux data using bands [2,1,0] for RGB."""
    n_bands = flux.shape[0]
    indices = [min(2, n_bands-1), min(1, n_bands-1), 0]
    rgb = np.stack([flux[i] for i in indices], axis=-1)
    for i in range(3):
        vmin, vmax = np.percentile(rgb[..., i], percentile)
        rgb[..., i] = np.clip((rgb[..., i] - vmin) / (vmax - vmin + 1e-10), 0, 1)
    return rgb

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(12, 6))

hsc_flux = np.array(batch['hsc_image']['flux'])
ls_flux = np.array(batch['legacysurvey_image']['flux'])

for i in range(4):
    axes[0, i].imshow(make_rgb(hsc_flux[i]), origin='lower')
    axes[0, i].set_title(f"HSC")
    axes[0, i].axis('off')
    
    axes[1, i].imshow(make_rgb(ls_flux[i]), origin='lower')
    axes[1, i].set_title(f"Legacy Survey")
    axes[1, i].axis('off')

plt.suptitle("Cross-matched HSC x Legacy Survey")
plt.tight_layout()
plt.show()